# MixHop: Higher-Order Neighborhood Convolution on Cora

Node Classification on Cora (Planetoid): Simultaneous mixing of 0-hop, 1-hop, and multi-hop neighborhood features. This notebook implements the approach with `MixHopConv` inside a `K3MixHop` model, trained with the Adam optimizer for 100 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `MixHopConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "MixHop: Higher-Order Neighborhood Convolution on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. MixHop Model Definition
class K3MixHop(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.MixHopConv(in_channels, hidden_channels, powers=[0, 1, 2])
        self.conv2 = k3_layers.MixHopConv(hidden_channels * 3, out_channels, powers=[0, 1])
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.tanh(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3MixHop(num_features, 32, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Generator & Training
def graph_data_generator():
    x = ops.convert_to_tensor(data.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data.edge_index, dtype="int64")
    y = ops.convert_to_tensor(data.y, dtype="int64")
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield (x, edge_index), y, mask

print(f"Training K3-Node MixHop on {backend} backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=100,
    verbose=1,
)

# 5. Evaluation
out = k3_model((data.x, data.edge_index))
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node MixHop execution completed successfully!")